# Assignment No: 02 - Cloud ETL Pipeline with Apache Spark
## Distributed Ingestion from Amazon S3 ➔ PySpark Transformations ➔ Amazon RDS / Cloud DB

**Student / Engineer:** Data Engineering Team  
**Technology Stack:** Apache Spark (PySpark), Amazon Web Services (S3, RDS), Python 3.14, SQLAlchemy  

---

## 1. Executive Summary & Architecture

This project implements an end-to-end cloud-native **Extract, Transform, and Load (ETL)** pipeline. The architecture leverages:
1. **Cloud Storage Layer (Amazon S3)**: Hosts raw transactional datasets and stores processed data lake exports.
2. **Distributed Compute Engine (Apache Spark / PySpark)**: Executes schema ingestion, statistical median/mean imputation, deduplication, type casting, quality validation gates, and analytical metric aggregation.
3. **Cloud Database Layer (Amazon RDS / Cloud DB)**: Persists validated fact tables, market KPI summaries, and automated audit logs.

```mermaid
graph LR
    subgraph AWS Cloud Storage
        S3Raw[Amazon S3: raw_transactions.csv]
    end
    subgraph PySpark Distributed Engine
        E[S3 Extractor] --> T1[Data Sanitization & Stripping]
        T1 --> T2[Statistical Imputation - Median/Mean/Mode]
        T2 --> T3[Deduplication & Quality Gates]
        T3 --> T4[Feature Engineering & Market KPIs]
    end
    subgraph AWS Cloud Database (RDS)
        T3 -->|Batch Load| RDS1[(RDS: clean_transactions)]
        T4 -->|Batch Load| RDS2[(RDS: kpi_market_summary)]
        T3 -->|Audit Metrics| RDS3[(RDS: data_quality_audit_log)]
    end
    S3Raw --> E
```


## 2. Environment Setup & Configuration
We initialize configuration constants, AWS credentials, and import the required libraries.

In [ ]:
import os
import sys
from pathlib import Path
import boto3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tabulate import tabulate

# PySpark Imports
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType, DateType

# Pipeline modules
import config
from s3_manager import S3Manager
from db_loader import CloudDBLoader
from spark_transform import get_spark_session, extract_raw_data, clean_and_transform_pipeline, audit_data_quality

print("Libraries successfully loaded!")
print(f"Target S3 Bucket: {config.AWS_S3_BUCKET}")
print(f"Target Database: {config.DB_ENGINE.upper()} ({config.DB_NAME})")

## 3. Step 1: Cloud Storage Operations (Amazon S3)
We verify the Amazon S3 bucket, upload our dirty raw dataset, and stream the raw file down for distributed processing.

In [ ]:
# Initialize S3 Storage Manager
s3 = S3Manager()
s3.ensure_bucket_exists()

# Upload raw transactions to S3
s3.upload_raw_dataset(config.SAMPLE_DATA_FILE, config.AWS_S3_RAW_KEY)

# List S3 Bucket Objects to confirm cloud persistence
s3_objects = s3.list_bucket_objects()
for obj in s3_objects:
    print(f"• S3 Key: {obj['Key']} ({obj['Size']} bytes)")

## 4. Step 2: Apache Spark Session & Raw Ingestion
We create an optimized PySpark session and ingest the raw dataset with `inferSchema=False` to prevent data loss.

In [ ]:
# Initialize Spark Session
spark = get_spark_session("Notebook_Spark_ETL")

# Download raw file from S3 to local staging
staging_path = config.OUTPUT_DIR / "staged_from_s3_raw.csv"
s3.download_to_local(config.AWS_S3_RAW_KEY, staging_path)

# Read into PySpark DataFrame
raw_df = extract_raw_data(spark, str(staging_path))
print(f"Raw Ingested Rows: {raw_df.count()}, Columns: {len(raw_df.columns)}")
raw_df.show(5, truncate=False)

## 5. Step 3: PySpark Distributed Transformations & Quality Auditing
Here we apply:
- **Missing Value Imputation**: Median for `age`, Mean for `annual_income`, `purchase_amount`, `store_rating`, and Mode for `gender`.
- **Deduplication**: Purging duplicate transactions by primary key.
- **Data Type Standardization**: Stripping currency symbols (`$`), commas, and enforcing strict schemas.
- **Feature Engineering**: Creating customer segments and demographic categories.
- **Quality Gates**: Asserting 0 nulls remain.

In [ ]:
# Execute transformation pipeline
clean_df, kpi_df, audit_report = clean_and_transform_pipeline(spark, raw_df)

print("=== PRE-CLEANSING AUDIT ===")
print(f"Total Rows: {audit_report['raw']['total_rows']}")
print(f"Duplicates Purged: {audit_report['raw']['duplicates_count']}")
print(f"Total Missing Values: {audit_report['raw']['total_nulls']}")

print("\n=== POST-CLEANSING AUDIT ===")
print(f"Total Cleaned Rows: {audit_report['clean']['total_rows']}")
print(f"Remaining Nulls: {audit_report['clean']['total_nulls']}")
print(f"Data Quality Improvement Index (DQII): {audit_report['dqii_percent']}%")

print("\n--- Sample Cleansed Transactions ---")
clean_df.select("transaction_id", "customer_name", "age", "gender", "annual_income", "purchase_amount", "city", "customer_segment").show(5)

## 6. Step 4: Loading into Amazon RDS / Cloud Database
We persist the validated fact table and aggregated analytical summaries into the cloud database tables.

In [ ]:
# Initialize Cloud DB Loader
db_loader = CloudDBLoader()
db_loader.test_connection()

# Load Tables into Database
db_loader.load_dataframe(clean_df, "clean_transactions", if_exists="replace")
db_loader.load_dataframe(kpi_df, "kpi_market_summary", if_exists="replace")
db_loader.log_pipeline_audit(audit_report, status="SUCCESS")

# Query back loaded tables
db_clean = db_loader.fetch_table_summary("clean_transactions")
db_kpi = db_loader.fetch_table_summary("kpi_market_summary")
db_audit = db_loader.fetch_table_summary("data_quality_audit_log")

print(f"Database 'clean_transactions' row count: {len(db_clean)}")
print(f"Database 'kpi_market_summary' row count: {len(db_kpi)}")
print(f"Database 'data_quality_audit_log' row count: {len(db_audit)}")

## 7. Step 5: Data Quality & Business Insights Dashboards
We visualize the data quality improvements and executive sales analytics.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Visual 1: Missing Value Profile
raw_nulls = audit_report['raw']['null_counts']
sns.barplot(x=list(raw_nulls.values()), y=list(raw_nulls.keys()), ax=axes[0], palette='Reds_r')
axes[0].set_title("Pre-Cleansing Missing Value Distribution")
axes[0].set_xlabel("Missing Count")

# Visual 2: Revenue by City Market
sns.barplot(data=db_kpi.sort_values(by='total_revenue', ascending=False).head(6), x='total_revenue', y='city', ax=axes[1], palette='Blues_r')
axes[1].set_title("Top Markets by Net Revenue ($)")
axes[1].set_xlabel("Total Revenue ($)")

plt.tight_layout()
plt.show()

## 8. Summary & Conclusion
- **Cloud Extraction**: Amazon S3 raw object streaming verified.
- **Distributed Cleansing**: PySpark imputed 23 missing values and purged duplicate records, achieving 100% completeness.
- **Cloud Storage & Loading**: Cleansed fact tables and market KPIs loaded into cloud database tables with full audit logs.
- **Spark Session**: Closed gracefully.

In [ ]:
spark.stop()
print("Spark Session stopped successfully. Pipeline Complete!")